<a href="https://colab.research.google.com/github/Gabriela-Sol/VpC2---Deteccion-de-humo-y-fuego/blob/integration%2Fmodelos/notebooks/03_entrenamiento_YOLO11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 - Entrenamiento YOLO11 para detección de humo y fuego

## Salidas esperadas

- pesos del modelo (`best.pt` y `last.pt`) en Drive;
- métricas de entrenamiento y validación;
- curvas de desempeño;
- matriz de confusión;
- carpeta de resultados asociada al experimento, en Drive.

In [1]:
# ============================================================
# Setup general del entorno
# ============================================================

from pathlib import Path
import os
import sys
import time
import shutil
import yaml
import torch
import pandas as pd

IN_COLAB = "google.colab" in sys.modules

print("Ejecutando en Google Colab:", IN_COLAB)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No se detectó GPU.")

Ejecutando en Google Colab: True
CUDA disponible: True
GPU: Tesla T4


In [2]:
# ============================================================
# Repositorio
# ============================================================

REPO_URL = "https://github.com/Gabriela-Sol/VpC2---Deteccion-de-humo-y-fuego"
REPO_NAME = "VpC2---Deteccion-de-humo-y-fuego"
TARGET_BRANCH = "integration/modelos"

PROJECT_DIR = Path("/content") / REPO_NAME

if IN_COLAB:
    if not PROJECT_DIR.exists():
        %cd /content
        !git clone --branch {TARGET_BRANCH} --single-branch {REPO_URL}.git

    %cd {PROJECT_DIR}
else:
    PROJECT_DIR = Path.cwd()

print("Proyecto:", PROJECT_DIR)

print("\nBranch actual:")
!git branch --show-current

/content
Cloning into 'VpC2---Deteccion-de-humo-y-fuego'...
remote: Enumerating objects: 750, done.
remote: Counting objects: 100% (123/123), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 750 (delta 94), reused 78 (delta 65), pack-reused 627 (from 3)
Receiving objects: 100% (750/750), 25.53 MiB | 163.00 KiB/s, done.
Resolving deltas: 100% (393/393), done.
/content/VpC2---Deteccion-de-humo-y-fuego
Proyecto: /content/VpC2---Deteccion-de-humo-y-fuego

Branch actual:
integration/modelos


In [3]:
# ============================================================
# Instalación de dependencias
# ============================================================

if IN_COLAB:
    !pip install -q -r {PROJECT_DIR / "requirements.txt"}

print("Dependencias instaladas.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 5.9 MB/s eta 0:00:00
Dependencias instaladas.


In [4]:
# ============================================================
# Montar Google Drive
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/VCII_DFire")
DRIVE_RUNS_DIR = DRIVE_PROJECT_DIR / "runs"
DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("Carpeta principal en Drive:", DRIVE_PROJECT_DIR)
print("Carpeta de corridas:", DRIVE_RUNS_DIR)

Mounted at /content/drive
Carpeta principal en Drive: /content/drive/MyDrive/VCII_DFire
Carpeta de corridas: /content/drive/MyDrive/VCII_DFire/runs


In [5]:
# ============================================================
# Descarga y localización del dataset D-Fire (Kaggle)
# ============================================================

import kagglehub

DATASET_ID = "sayedgamal99/smoke-fire-detection-yolo"

dataset_root = Path(kagglehub.dataset_download(DATASET_ID))

print("Dataset descargado/localizado en:")
print(dataset_root)


def find_yolo_dataset_dir(root: Path) -> Path:
    """
    Busca automáticamente la carpeta que contiene la estructura esperada:
    train/images, train/labels, val/images, val/labels.
    """
    candidates = [root] + [p for p in root.rglob("*") if p.is_dir()]

    for candidate in candidates:
        required = [
            candidate / "train" / "images",
            candidate / "train" / "labels",
            candidate / "val" / "images",
            candidate / "val" / "labels",
        ]

        if all(path.exists() for path in required):
            return candidate

    raise FileNotFoundError(
        "No se encontró una estructura YOLO válida con train/images, train/labels, val/images y val/labels."
    )


DATA_DIR = find_yolo_dataset_dir(dataset_root)

print("Carpeta de datos YOLO detectada:")
print(DATA_DIR)

for split in ["train", "val", "test"]:
    split_dir = DATA_DIR / split
    print(f"{split}: existe={split_dir.exists()} -> {split_dir}")

Using Colab cache for faster access to the 'smoke-fire-detection-yolo' dataset.
Dataset descargado/localizado en:
/kaggle/input/smoke-fire-detection-yolo
Carpeta de datos YOLO detectada:
/kaggle/input/smoke-fire-detection-yolo/data
train: existe=True -> /kaggle/input/smoke-fire-detection-yolo/data/train
val: existe=True -> /kaggle/input/smoke-fire-detection-yolo/data/val
test: existe=True -> /kaggle/input/smoke-fire-detection-yolo/data/test


In [6]:
# ============================================================
# Generación del YAML de dataset para YOLO
# ============================================================

DFIRE_YAML = Path("/content/dfire_colab.yaml")

dfire_config = {
    "path": str(DATA_DIR),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "nc": 2,
    "names": {
        0: "smoke",
        1: "fire",
    },
}

with open(DFIRE_YAML, "w", encoding="utf-8") as file:
    yaml.safe_dump(dfire_config, file, sort_keys=False, allow_unicode=True)

print("Archivo YAML de dataset creado en:")
print(DFIRE_YAML)

print("\nContenido:")
with open(DFIRE_YAML, "r", encoding="utf-8") as file:
    print(file.read())

Archivo YAML de dataset creado en:
/content/dfire_colab.yaml

Contenido:
path: /kaggle/input/smoke-fire-detection-yolo/data
train: train/images
val: val/images
test: test/images
nc: 2
names:
  0: smoke
  1: fire



In [7]:
# ============================================================
# Config del experimento (equivalente a
# configs/experiments/yolo11n_baseline.yaml, definida inline)
# ============================================================

experiment_config = {
    "experiment": {
        "name": "yolo11n_baseline",
        "family": "YOLO11",
        "model": "yolo11n.pt",
        "description": "Baseline liviano con YOLO11 nano.",
    },
    "training": {
        "epochs": 50,
        "imgsz": 640,
        "batch": 16,
        "patience": 10,
        "optimizer": "auto",
        "lr0": 0.01,
        "seed": 42,
    },
    "output": {
        "project": str(DRIVE_RUNS_DIR),
        "save_weights": True,
    },
}

experiment_name = experiment_config["experiment"]["name"]
model_name = experiment_config["experiment"]["model"]
family = experiment_config["experiment"]["family"]
print("Experimento:", experiment_name)
print("Modelo:", model_name)
print(experiment_config)

Experimento: yolo11n_baseline
Modelo: yolo11n.pt
{'experiment': {'name': 'yolo11n_baseline', 'family': 'YOLO11', 'model': 'yolo11n.pt', 'description': 'Baseline liviano con YOLO11 nano.'}, 'training': {'epochs': 50, 'imgsz': 640, 'batch': 16, 'patience': 10, 'optimizer': 'auto', 'lr0': 0.01, 'seed': 42}, 'output': {'project': '/content/drive/MyDrive/VCII_DFire/runs', 'save_weights': True}}


In [8]:
# ============================================================
# Función de entrenamiento a partir de la config
# ============================================================

from ultralytics import YOLO
import pandas as pd


def train_from_config(config: dict, data_yaml: Path) -> dict:
    """
    Entrena un modelo YOLO a partir de un diccionario de configuración.
    Los resultados se guardan en una carpeta específica por experimento,
    dentro de Google Drive.
    """
    exp = config["experiment"]
    train_cfg = config["training"]
    output_cfg = config["output"]

    experiment_name = exp["name"]
    model_name = exp["model"]
    project_dir = Path(output_cfg["project"])

    project_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 80)
    print(f"Experimento: {experiment_name}")
    print(f"Familia: {exp['family']}")
    print(f"Modelo: {model_name}")
    print(f"Resultados en: {project_dir / experiment_name}")
    print("=" * 80)

    start_time = time.time()

    model = YOLO(model_name)

    results = model.train(
        data=str(data_yaml),
        epochs=train_cfg["epochs"],
        imgsz=train_cfg["imgsz"],
        batch=train_cfg["batch"],
        patience=train_cfg["patience"],
        optimizer=train_cfg["optimizer"],
        lr0=train_cfg["lr0"],
        seed=train_cfg["seed"],
        project=str(project_dir),
        name=experiment_name,
        save_period=train_cfg.get("save_period", 1),
        exist_ok=True,
        plots=True,
    )

    elapsed_time = time.time() - start_time

    experiment_dir = project_dir / experiment_name
    best_model_path = experiment_dir / "weights" / "best.pt"
    last_model_path = experiment_dir / "weights" / "last.pt"
    results_csv = experiment_dir / "results.csv"

    summary = {
        "experiment": experiment_name,
        "family": exp["family"],
        "model": model_name,
        "epochs": train_cfg["epochs"],
        "imgsz": train_cfg["imgsz"],
        "batch": train_cfg["batch"],
        "training_time_min": round(elapsed_time / 60, 2),
        "experiment_dir": str(experiment_dir),
        "best_model_path": str(best_model_path),
        "last_model_path": str(last_model_path),
        "results_csv": str(results_csv),
        "best_exists": best_model_path.exists(),
        "last_exists": last_model_path.exists(),
    }

    print("\nEntrenamiento finalizado.")
    print("Tiempo total [min]:", summary["training_time_min"])
    print("Best model:", best_model_path)
    print("Last model:", last_model_path)

    return summary

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [9]:
# ============================================================
# Ejecución opcional del entrenamiento
# ============================================================

RUN_TRAINING = False

if RUN_TRAINING:
    experiment_summary = train_from_config(
        config=experiment_config,
        data_yaml=DFIRE_YAML,
    )

    display(pd.DataFrame([experiment_summary]))

else:
    print("Entrenamiento omitido.")
    print("Se utilizarán los artefactos del experimento ya entrenado.")

Entrenamiento omitido.
Se utilizarán los artefactos del experimento ya entrenado.


## Resultados del experimento final

Para el análisis se utiliza la corrida final de YOLO11n entrenada durante 50 épocas. A partir de los artefactos almacenados en Google Drive se recuperan la configuración y los resultados necesarios para la comparación entre modelos.

In [10]:
# ============================================================
# Artefactos del experimento final
# ============================================================

FINAL_RUN_DIR = DRIVE_RUNS_DIR / "yolo11n_baseline"

ARGS_PATH = FINAL_RUN_DIR / "args.yaml"
RESULTS_CSV = FINAL_RUN_DIR / "results.csv"
BEST_PT = FINAL_RUN_DIR / "weights" / "best.pt"

required_files = [
    ARGS_PATH,
    RESULTS_CSV,
    BEST_PT,
]

missing = [path for path in required_files if not path.exists()]

if missing:
    print("Faltan:")
    for path in missing:
        print("-", path)
else:
    print("Corrida encontrada:", FINAL_RUN_DIR)

Corrida encontrada: /content/drive/MyDrive/VCII_DFire/runs/yolo11n_baseline


In [11]:
with open(ARGS_PATH, "r", encoding="utf-8") as file:
    final_args = yaml.safe_load(file)

history = pd.read_csv(RESULTS_CSV)
history.columns = history.columns.str.strip()

print("Corrida:", FINAL_RUN_DIR.name)
print("Épocas configuradas:", final_args["epochs"])
print("Épocas registradas:", len(history))
print("imgsz:", final_args["imgsz"])
print("batch:", final_args["batch"])
print("best.pt:", BEST_PT.exists())

Corrida: yolo11n_baseline
Épocas configuradas: 50
Épocas registradas: 50
imgsz: 640
batch: 16
best.pt: True


In [12]:
# ============================================================
# Metadata del entrenamiento
# ============================================================

from ultralytics import YOLO

model = YOLO(str(BEST_PT))

params_M = sum(
    parameter.numel()
    for parameter in model.model.parameters()
) / 1e6

best_idx = history["metrics/mAP50-95(B)"].idxmax()
best_row = history.loc[best_idx]

best_epoch = int(best_row["epoch"])

times = history["time"].astype(float).to_numpy()

total_seconds = times[0]

for i in range(1, len(times)):
    delta = times[i] - times[i - 1]

    if delta >= 0:
        total_seconds += delta
    else:
        total_seconds += times[i]

train_time_min = total_seconds / 60

print(f"Modelo: YOLO11n")
print(f"Parámetros: {params_M:.2f} M")
print(f"Épocas: {final_args['epochs']}")
print(f"Mejor época según mAP@0.5:0.95: {best_epoch}")
print(f"Tiempo registrado de entrenamiento: {train_time_min:.2f} min")

Modelo: YOLO11n
Parámetros: 2.59 M
Épocas: 50
Mejor época según mAP@0.5:0.95: 50
Tiempo registrado de entrenamiento: 251.10 min


## Evaluación del modelo final

Se evalúa el mejor checkpoint sobre el conjunto de validación de D-Fire para obtener métricas globales, métricas por clase y velocidad de inferencia.

In [13]:
# ============================================================
# Evaluación sobre test
# ============================================================

device = 0 if torch.cuda.is_available() else "cpu"

metrics = model.val(
    data=str(DFIRE_YAML),
    split="test",
    imgsz=int(final_args["imgsz"]),
    batch=int(final_args["batch"]),
    device=device,
    plots=True,
    project=str(DRIVE_RUNS_DIR),
    name="yolo11n_baseline_val_final",
    exist_ok=True,
)

print("Resultados de test:")
print(metrics.save_dir)

Ultralytics 8.4.120 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,582,542 parameters, 0 gradients, 6.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 21.8±10.9 MB/s, size: 227.8 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /kaggle/input/smoke-fire-detection-yolo/data/test/labels... 4295 images, 2005 backgrounds, 15 corrupt: 100% ━━━━━━━━━━━━ 4306/4306 86.2it/s 50.0s
val: /kaggle/input/smoke-fire-detection-yolo/data/test/images/WEB10769.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0297]
val: /kaggle/input/smoke-fire-detection-yolo/data/test/images/WEB10775.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0156]
val: /kaggle/input/smoke-fire-detection-yolo/data/test/images/WEB11243.jpg: ignoring corrupt image/label: [

In [14]:
# ============================================================
# Métricas por clase
# ============================================================

class_metrics = pd.DataFrame(
    metrics.summary(decimals=5)
)

display(class_metrics)

,Class,Images,Instances,Box-P,Box-R,Box-F1,mAP50,mAP50-95
0,smoke,2066,2298,0.80254,0.76284,0.78219,0.82005,0.50558
1,fire,1111,2868,0.70239,0.61636,0.65657,0.68525,0.36021


In [15]:
# ============================================================
# Resumen estandarizado del experimento
# ============================================================

results_dict = metrics.results_dict

precision = float(results_dict["metrics/precision(B)"])
recall = float(results_dict["metrics/recall(B)"])

f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall > 0
    else 0.0
)

class_results = class_metrics.set_index("Class")

smoke = class_results.loc["smoke"]
fire = class_results.loc["fire"]

inference_ms = float(metrics.speed["inference"])

fps = (
    1000.0 / inference_ms
    if inference_ms > 0
    else float("nan")
)

device_name = (
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU"
)

summary = pd.DataFrame([{
    "experiment": experiment_name,
    "family": family,
    "model": model_name,

    "params_M": round(params_M, 2),
    "epochs": int(final_args["epochs"]),
    "imgsz": int(final_args["imgsz"]),
    "batch": int(final_args["batch"]),
    "train_time_min": round(train_time_min, 2),

    "mAP50": float(results_dict["metrics/mAP50(B)"]),
    "mAP50_95": float(results_dict["metrics/mAP50-95(B)"]),

    "precision": precision,
    "recall": recall,
    "f1": f1,

    "mAP50_smoke": float(smoke["mAP50"]),
    "mAP50_fire": float(fire["mAP50"]),

    "mAP50_95_smoke": float(smoke["mAP50-95"]),
    "mAP50_95_fire": float(fire["mAP50-95"]),

    "fps": round(fps, 2),
    "device": device_name,
    "split": "val",

    "best_epoch": best_epoch,
}])

display(summary)

,experiment,family,model,params_M,epochs,imgsz,batch,train_time_min,mAP50,mAP50_95,...,recall,f1,mAP50_smoke,mAP50_fire,mAP50_95_smoke,mAP50_95_fire,fps,device,split,best_epoch
0,yolo11n_baseline,YOLO11,yolo11n.pt,2.59,50,640,16,251.1,0.752648,0.432893,...,0.689601,0.719663,0.82005,0.68525,0.50558,0.36021,373.87,Tesla T4,val,50


In [ ]:
# ============================================================
# Exportación de resultados
# ============================================================

REPORTS_RESULTS_DIR = (
    PROJECT_DIR
    / "reports"
    / "results"
    / experiment_name
)

REPORTS_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

summary.to_csv(
    REPORTS_RESULTS_DIR / "metrics_summary.csv",
    index=False,
)

# Artefactos del entrenamiento
for filename in [
    "results.csv",
    "results.png",
    "args.yaml",
]:
    src = FINAL_RUN_DIR / filename

    if src.exists():
        shutil.copy2(
            src,
            REPORTS_RESULTS_DIR / filename,
        )

# Figuras de la validación del best.pt
VALIDATION_DIR = Path(metrics.save_dir)

for filename in [
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "BoxPR_curve.png",
    "BoxF1_curve.png",
    "BoxP_curve.png",
    "BoxR_curve.png",
]:
    src = VALIDATION_DIR / filename

    if src.exists():
        shutil.copy2(
            src,
            REPORTS_RESULTS_DIR / filename,
        )

# Configuración utilizada
with open(REPORTS_RESULTS_DIR / "experiment_config_used.yaml", "w", encoding="utf-8") as file:
    yaml.safe_dump(experiment_config, file, sort_keys=False, allow_unicode=True)

print("Resultados guardados en:")
print(REPORTS_RESULTS_DIR)

print("\nArchivos:")
for path in sorted(REPORTS_RESULTS_DIR.iterdir()):
    print("-", path.name)

In [ ]:
SUMMARY_PATH = (
    PROJECT_DIR
    / "reports"
    / "results"
    / "yolo11n_baseline"
    / "metrics_summary.csv"
)

print("Existe:", SUMMARY_PATH.exists())

if SUMMARY_PATH.exists():
    tmp = pd.read_csv(SUMMARY_PATH)
    print(tmp.columns.tolist())
    display(tmp)

##############################

In [17]:
# ============================================================
# Artefactos disponibles del experimento
# ============================================================

RESULTS_DIR = (
    PROJECT_DIR
    / "reports"
    / "results"
    / "yolo11n_baseline"
)

ARGS_PATH = RESULTS_DIR / "args.yaml"
RESULTS_CSV = RESULTS_DIR / "results.csv"

required_files = [
    ARGS_PATH,
    RESULTS_CSV,
]

missing = [path for path in required_files if not path.exists()]

if missing:
    raise FileNotFoundError(
        "Faltan artefactos del experimento:\n"
        + "\n".join(str(path) for path in missing)
    )

with open(ARGS_PATH, "r", encoding="utf-8") as file:
    final_args = yaml.safe_load(file)

history = pd.read_csv(RESULTS_CSV)
history.columns = history.columns.str.strip()

print("Épocas configuradas:", final_args["epochs"])
print("Épocas registradas:", len(history))
print("imgsz:", final_args["imgsz"])
print("batch:", final_args["batch"])

Épocas configuradas: 50
Épocas registradas: 50
imgsz: 640
batch: 16


In [18]:
# ============================================================
# Métricas y metadata del entrenamiento
# ============================================================

best_idx = history["metrics/mAP50-95(B)"].idxmax()
best_row = history.loc[best_idx]

best_epoch = int(best_row["epoch"])

precision = float(best_row["metrics/precision(B)"])
recall = float(best_row["metrics/recall(B)"])
map50 = float(best_row["metrics/mAP50(B)"])
map50_95 = float(best_row["metrics/mAP50-95(B)"])

f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall > 0
    else 0.0
)

times = history["time"].astype(float).to_numpy()

total_seconds = times[0]

for i in range(1, len(times)):
    delta = times[i] - times[i - 1]

    if delta >= 0:
        total_seconds += delta
    else:
        total_seconds += times[i]

train_time_min = total_seconds / 60

print(f"Mejor época: {best_epoch}")
print(f"Precision: {precision:.5f}")
print(f"Recall: {recall:.5f}")
print(f"F1: {f1:.5f}")
print(f"mAP@0.5: {map50:.5f}")
print(f"mAP@0.5:0.95: {map50_95:.5f}")
print(f"Tiempo registrado: {train_time_min:.2f} min")

Mejor época: 50
Precision: 0.77699
Recall: 0.69131
F1: 0.73165
mAP@0.5: 0.76256
mAP@0.5:0.95: 0.44491
Tiempo registrado: 251.10 min


In [19]:
# ============================================================
# Resumen estandarizado del experimento
# ============================================================

summary = pd.DataFrame([{
    "experiment": "yolo11n_baseline",
    "family": "YOLO11",
    "model": "YOLO11n",

    "params_M": pd.NA,
    "epochs": int(final_args["epochs"]),
    "imgsz": int(final_args["imgsz"]),
    "batch": int(final_args["batch"]),
    "train_time_min": round(train_time_min, 2),

    "mAP50": map50,
    "mAP50_95": map50_95,

    "precision": precision,
    "recall": recall,
    "f1": f1,

    "mAP50_smoke": pd.NA,
    "mAP50_fire": pd.NA,
    "mAP50_95_smoke": pd.NA,
    "mAP50_95_fire": pd.NA,

    "fps": pd.NA,
    "device": pd.NA,
    "split": "val",

    "best_epoch": best_epoch,
}])

display(summary)

summary.to_csv(
    RESULTS_DIR / "metrics_summary.csv",
    index=False,
)

,experiment,family,model,params_M,epochs,imgsz,batch,train_time_min,mAP50,mAP50_95,...,recall,f1,mAP50_smoke,mAP50_fire,mAP50_95_smoke,mAP50_95_fire,fps,device,split,best_epoch
0,yolo11n_baseline,YOLO11,YOLO11n,<NA>,50,640,16,251.1,0.76256,0.44491,...,0.69131,0.73165,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,val,50
